# Silver Layer: Clean and Transform NSE Data
This notebook reads Bronze Delta data, validates the raw rows, applies cleanup logic, and writes the cleaned dataset into the Silver layer.

In [0]:
%pip install pyyaml
import logging
import yaml
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, lit, to_date

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("silver_transform")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Load Configuration
Load the Silver configuration file and storage paths.

In [0]:
# Configuration for Silver transformation
bronze_table = "dw_stocksight.bronze_stocks_nse.nse_stock_details"
silver_table = "dw_stocksight.silver_stocks_nse.nse_stock_details"
partition_col = "ingestion_date"

logger.info(f"Bronze source: {bronze_table}")
logger.info(f"Silver target: {silver_table}")
logger.info(f"Partition column: {partition_col}")

## Initialize Spark Session
Create or attach to a Spark session configured for Delta Lake.

In [0]:
spark = SparkSession.builder \
    .appName("Silver_NSE_Transform") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

logger.info("Spark session initialized")

## Read Bronze Data
Read the raw Bronze Delta table or path and review the schema.

In [0]:
# Read Bronze managed table
df_bronze = spark.table(bronze_table)
logger.info(f"Loaded Bronze table: {bronze_table}, Row count: {df_bronze.count()}")

df_bronze.printSchema()
df_bronze.show(5, truncate=False)

root
 |-- priority: long (nullable = true)
 |-- symbol: string (nullable = true)
 |-- identifier: string (nullable = true)
 |-- open: double (nullable = true)
 |-- dayHigh: double (nullable = true)
 |-- dayLow: double (nullable = true)
 |-- lastPrice: double (nullable = true)
 |-- previousClose: double (nullable = true)
 |-- change: double (nullable = true)
 |-- pChange: double (nullable = true)
 |-- ffmc: double (nullable = true)
 |-- yearHigh: double (nullable = true)
 |-- yearLow: double (nullable = true)
 |-- totalTradedVolume: long (nullable = true)
 |-- stockIndClosePrice: long (nullable = true)
 |-- totalTradedValue: double (nullable = true)
 |-- lastUpdateTime: string (nullable = true)
 |-- nearWKH: double (nullable = true)
 |-- nearWKL: double (nullable = true)
 |-- perChange365d: double (nullable = true)
 |-- perChange30d: double (nullable = true)
 |-- date365dAgo: string (nullable = true)
 |-- date30dAgo: string (nullable = true)
 |-- chartTodayPath: string (nullable = true)

## Clean and Transform
Drop raw-only columns, keep governance fields, and prepare the Silver schema.

In [0]:
columns_to_drop = [
    "priority",
    "identifier",
    "change",
    "pChange",
    "ffmc",
    "yearHigh",
    "yearLow",
    "nearWKH",
    "nearWKL",
    "perChange365d",
    "perChange30d",
    "date365dAgo",
    "date30dAgo",
    "chartTodayPath",
    "chart30dPath",
    "chart365dPath",
    "series",
    "meta_symbol",
    "meta_activeSeries",
    "meta_debtSeries",
    "meta_isCASec",
    "meta_isSLBSec",
    "meta_isDebtSec",
    "meta_isSuspended",
    "meta_tempSuspendedSeries",
    "meta_isMunicipalBond",
    "meta_isHybridSymbol",
    "meta_quotepreopenstatus_equityTime",
    "meta_quotepreopenstatus_preOpenTime",
    "meta_quotepreopenstatus_QuotePreOpenFlag"
]

clean_df = df_bronze.drop(*[c for c in columns_to_drop if c in df_bronze.columns])

clean_df = clean_df.withColumnRenamed("symbol", "stock_name")\
    .withColumnRenamed("lastPrice", "last_traded_price")\
    .withColumnRenamed("meta_companyName", "company_name")\
    .withColumnRenamed("meta_industry", "industry")\
    .withColumnRenamed("meta_isFNOSec", "is_FNO")\
    .withColumnRenamed("meta_isETFSec", "is_ETF")\
    .withColumnRenamed("meta_isDelisted", "is_Delisted")\
    .withColumnRenamed("meta_isin", "ISIN")\
    .withColumnRenamed("meta_slb_isin", "ISIN_SLB")\
    .withColumnRenamed("meta_listingDate", "listing_date")\
    .withColumnRenamed("meta_segment", "segment")


clean_df = clean_df.withColumn("processed_timestamp", current_timestamp())
clean_df = clean_df.withColumn(partition_col, to_date(col(partition_col)))

clean_df.printSchema()
clean_df.show(10, truncate=False)

root
 |-- stock_name: string (nullable = true)
 |-- open: double (nullable = true)
 |-- dayHigh: double (nullable = true)
 |-- dayLow: double (nullable = true)
 |-- last_traded_price: double (nullable = true)
 |-- previousClose: double (nullable = true)
 |-- totalTradedVolume: long (nullable = true)
 |-- stockIndClosePrice: long (nullable = true)
 |-- totalTradedValue: double (nullable = true)
 |-- lastUpdateTime: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- is_FNO: boolean (nullable = true)
 |-- is_ETF: boolean (nullable = true)
 |-- is_Delisted: boolean (nullable = true)
 |-- ISIN: string (nullable = true)
 |-- ISIN_SLB: string (nullable = true)
 |-- listing_date: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- pipeline_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- file_name: string (nullable = true)
 |-- 

## Write Silver Delta
Write the cleaned dataset to the Silver Delta path and register the table.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS dw_stocksight.silver_stocks_nse")

clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy(partition_col) \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table)

logger.info(f"Written Silver managed Delta table: {silver_table}")
display(clean_df.limit(5))


stock_name,open,dayHigh,dayLow,last_traded_price,previousClose,totalTradedVolume,stockIndClosePrice,totalTradedValue,lastUpdateTime,company_name,industry,is_FNO,is_ETF,is_Delisted,ISIN,ISIN_SLB,listing_date,segment,ingestion_timestamp,pipeline_run_id,source_system,file_name,ingestion_date,processed_timestamp
NIFTY TOTAL MARKET,12790.7,12802.8,12656.05,12761.5,12864.6,3469710725,0,1.28028689671287E12,30-Apr-2026 16:00:00,null,null,null,null,null,null,null,null,null,2026-05-03T07:53:19.793Z,20260428_run01,NSE,nse_raw_20260503_075319.json,2026-05-03,2026-05-03T07:54:36.126Z
RELIANCE,1409.0,1437.0,1393.1,1436.0,1425.4,30957881,0,4.384936180602E10,null,Reliance Industries Limited,Refineries & Marketing,true,false,false,INE002A01018,INE002A01018,1995-11-29,EQUITY,2026-05-03T07:53:19.793Z,20260428_run01,NSE,nse_raw_20260503_075319.json,2026-05-03,2026-05-03T07:54:36.126Z
HDFCBANK,770.0,778.8,762.25,774.75,779.0,47998755,0,3.69763209018E10,null,HDFC Bank Limited,Private Sector Bank,true,false,false,INE040A01034,INE040A01034,1995-11-08,EQUITY,2026-05-03T07:53:19.793Z,20260428_run01,NSE,nse_raw_20260503_075319.json,2026-05-03,2026-05-03T07:54:36.126Z
SYNGENE,431.95,518.55,429.9,468.05,432.15,59274654,0,2.861009724618E10,null,Syngene International Limited,Healthcare Research Analytics & Technology,false,false,false,INE398R01022,INE398R01022,2015-08-11,EQUITY,2026-05-03T07:53:19.793Z,20260428_run01,NSE,nse_raw_20260503_075319.json,2026-05-03,2026-05-03T07:54:36.126Z
MEESHO,173.38,196.62,173.06,194.0,172.68,150163771,0,2.858217217214E10,null,Meesho Limited,E-Retail/ E-Commerce,false,false,false,INE0VDM01015,INE0VDM01015,2025-12-10,EQUITY,2026-05-03T07:53:19.793Z,20260428_run01,NSE,nse_raw_20260503_075319.json,2026-05-03,2026-05-03T07:54:36.126Z
